## Created from MODIS and VIIRS active fires for 2023 and 2024.
We downloaded the af files from NASA (https://firms.modaps.eosdis.nasa.gov/data/download/DL_FIRE_SV-C2_578623.zip) and the burned areas from effis. 
Then we:
 - Created a buffer of 1000 metres in the burned area polygons
 - Filtered to take only the summer months (May - September)
 - Selected only the af that intersect with the buffered fire polygons and deleted the rest
 - Performed spatial join between af and bsm polygons and compared their dates (if their distance in days was more than 4 days we deleted the af)

In [1]:
import xarray as xr
import autoroot
from satpy.scene import Scene
from pyhdf.SD import SD, SDC 
import pandas as pd
import cartopy.crs as ccrs
import rioxarray
import numpy as np
import matplotlib.pyplot as plt
import os
from datetime import datetime
import fnmatch
from pyproj import CRS, Transformer
from pyhdf.SD import SD, SDC 
from rs_tools._src.geoprocessing.match import match_timestamps_af
from rs_tools._src.utils.io import get_list_filenames
from pathlib import Path
from tqdm import tqdm


/home/sgirtsou/miniconda3/envs/rs_tools/lib/python3.11/site-packages/pyproj/__init__.py:95: UserWarning: pyproj unable to set database path.
  _pyproj_global_context_initialize()
/home/sgirtsou/miniconda3/envs/rs_tools/lib/python3.11/site-packages/goes2go/data.py:665: FutureWarning: 'H' is deprecated and will be removed in a future version. Please use 'h' instead of 'H'.
  within=pd.to_timedelta(config["nearesttime"].get("within", "1h")),
/home/sgirtsou/miniconda3/envs/rs_tools/lib/python3.11/site-packages/goes2go/NEW.py:188: FutureWarning: 'H' is deprecated and will be removed in a future version. Please use 'h' instead of 'H'.
  within=pd.to_timedelta(config["nearesttime"].get("within", "1h")),


In [2]:
year = 2024

In [3]:
def convert_lat_lon_to_x_y(crs, lon, lat):
    transformer = Transformer.from_crs(CRS("+proj=latlon"), crs, always_xy=True)
    x, y = transformer.transform(lon, lat)
    return x, y
def parse_af_dates_from_file(file):
    timestamp = Path(file).name.split("_")[0]
    return timestamp

In [4]:
msg_path = '/mnt/outputs/geoprocessed'
save_af_path = f'/mnt/data8tb/fire_detection/af_nasa_geoprocessed/{year}'
os.makedirs(save_af_path, exist_ok=True)

In [5]:
af = pd.read_csv(f'/home/sgirtsou/Projects/rs_tools/data/fire-detection/nasa_af_corrected_{str(year)}.csv')

/tmp/ipykernel_1417696/3697648076.py:1: DtypeWarning: Columns (17) have mixed types. Specify dtype option on import or set low_memory=False.
  af = pd.read_csv(f'/home/sgirtsou/Projects/rs_tools/data/fire-detection/nasa_af_corrected_{str(year)}.csv')


In [6]:
af.ACQ_DATE

0         2024-09-08
1         2024-09-07
2         2024-09-07
3         2024-09-08
4         2024-09-17
             ...    
101693    2024-09-27
101694    2024-09-27
101695    2024-09-27
101696    2024-09-27
101697    2024-09-28
Name: ACQ_DATE, Length: 101698, dtype: object

In [7]:
af['ACQ_DATETIME'] = pd.to_datetime(af['ACQ_DATE'] + ' ' + af['ACQ_TIME'].astype(str).str.zfill(4), format='%Y-%m-%d %H%M')
af['datetime'] = af['ACQ_DATETIME'].dt.strftime('%Y%m%d%H%M00')

In [8]:
af_sel = af[af['ACQ_DATETIME'].dt.year == year]

In [9]:
year

2024

In [10]:
af_sel.drop_duplicates(subset=['LATITUDE','LONGITUDE','ACQ_DATE', 'ACQ_TIME'], inplace=True)

In [11]:
unique_times_af = af_sel['ACQ_DATETIME'].dt.strftime('%Y%m%d%H%M00').unique().tolist()

In [12]:
files_msg_af = get_list_filenames(msg_path, ".nc", str(year))
unique_times_msg = list(set(map(parse_af_dates_from_file, files_msg_af)))
df_matches = match_timestamps_af(unique_times_af, unique_times_msg, cutoff=15)
df_matches.columns = ['timestamp_af', 'timestamp_msg']

No valid af mask found for 2024-09-08 00:43:00
No valid af mask found for 2024-09-07 23:26:00
No valid af mask found for 2024-09-08 12:05:00
No valid af mask found for 2024-09-17 01:15:00
No valid af mask found for 2024-09-17 02:55:00
No valid af mask found for 2024-09-17 02:56:00
No valid af mask found for 2024-09-17 12:38:00
No valid af mask found for 2024-09-17 14:19:00
No valid af mask found for 2024-09-18 02:37:00
No valid af mask found for 2024-09-18 12:20:00
No valid af mask found for 2024-09-18 14:00:00
No valid af mask found for 2024-09-19 02:18:00
No valid af mask found for 2024-09-19 13:41:00
No valid af mask found for 2024-09-17 01:37:00
No valid af mask found for 2024-09-17 03:19:00
No valid af mask found for 2024-09-17 13:01:00
No valid af mask found for 2024-09-18 01:20:00
No valid af mask found for 2024-09-18 03:00:00
No valid af mask found for 2024-09-18 12:44:00
No valid af mask found for 2024-09-18 14:25:00
No valid af mask found for 2024-09-19 02:43:00
No valid af m

In [13]:
df_matches

,timestamp_af,timestamp_msg


In [13]:
def create_fires_ds(df_af_datetime, msg):
    array = np.zeros((msg.y.size, msg.x.size))
    var = "msg_seviri_fes_3km"
    crs_wkt = msg[var].crs_wkt
    crs = CRS(crs_wkt)
    for index, row in df_af_datetime.iterrows():
        lon = row['LONGITUDE']
        lat = row['LATITUDE']
        x_sel, y_sel = convert_lat_lon_to_x_y(crs, lon, lat)
        selected = msg.sel(x=x_sel, y=y_sel, method='nearest')
        # Get the indices of the nearest point
        x_idx = msg.get_index('x').get_loc(selected['x'].item())
        y_idx = msg.get_index('y').get_loc(selected['y'].item())
        try:
            array[y_idx, x_idx] = 1
        except IndexError:
            print(f"Index out of bounds: y_idx={y_idx}, x_idx={x_idx}, array shape={array.shape}")
    da = xr.DataArray(
        array,
        coords={"y": msg.y, "x": msg.x},
        dims=("y", "x")
    )
    da.attrs['af_time'] = df_af_datetime.datetime.unique()[0]
    da.attrs['DAYNIGHT'] = df_af_datetime.DAYNIGHT.unique()[0]
    return da

In [14]:
for index, row in tqdm(df_matches.iterrows(), total=len(df_matches), desc="Processing files"):
    df_af_datetime = af_sel[af_sel.datetime == row['timestamp_af']]
    msg = xr.open_dataset(os.path.join(msg_path, f"{row.timestamp_msg}_msg.nc"))
    fires_ds = create_fires_ds(df_af_datetime, msg)
    fires_ds.to_netcdf(os.path.join(save_af_path, f"{row.timestamp_msg}_af.nc"))

Processing files: 100%|██████████| 2024/2024 [42:40<00:00,  1.27s/it]   


## After this I run the prepatcher for af

### prepatcher_af.py from VScode (not terminal)
    prepatch(read_path = '/mnt/data8tb/fire_detection/af_nasa_geoprocessed/2023/', save_path='/mnt/data8tb/fire_detection/af_nasa_patched/2023/', patch_size=32, stride_size=32, fire_cutoff=1, save_filetype='tif')


## Copy msg patches to have harmonized folders
### I did the copy from /mnt/data8tb/fire_detection/copy_msg_geoprocessed.sh

In [ ]:
missing = []

In [ ]:
import os
import shutil

# Define source and destination directories
af_dir = "/mnt/data8tb/fire_detection/af_nasa_geoprocessed/2023"
msg_dir = "/mnt/outputs/geoprocessed"
dest_dir = "/mnt/data8tb/fire_detection/msg_geoprocessed/2023"

# Get the list of _af.nc files
af_files = [f for f in os.listdir(af_dir) if f.endswith("nc")]

# Extract timestamps and copy corresponding _msg.nc files
for af_file in af_files:
    timestamp = af_file.split("_")[0]  # Extract timestamp (first part of the filename)
    msg_file = f"{timestamp}_msg.nc"
    if os.path.exists(os.path.join(dest_dir, msg_file)):  # Check if the _msg.nc file exists
        continue
    msg_path = os.path.join(msg_dir, msg_file)
    try:
        shutil.copy(msg_path, os.path.join(dest_dir, msg_file))  # Copy the file
        print(f"Copied: {msg_file}")
        time.sleep(0.1)
    except:
        missing.append(msg_file)
        print(f"Missing: {msg_file}")

Copied: 20230904095742_msg.nc
Missing: 20230904095742_msg.nc
Copied: 20230908121241_msg.nc
Missing: 20230908121241_msg.nc
Copied: 20230625104241_msg.nc
Missing: 20230625104241_msg.nc
Copied: 20230730001241_msg.nc
Missing: 20230730001241_msg.nc
Copied: 20230505142741_msg.nc
Missing: 20230505142741_msg.nc
Copied: 20230830124241_msg.nc
Missing: 20230830124241_msg.nc
Copied: 20230721212741_msg.nc
Missing: 20230721212741_msg.nc
Copied: 20230713012742_msg.nc
Missing: 20230713012742_msg.nc
Copied: 20230826024241_msg.nc
Missing: 20230826024241_msg.nc
Copied: 20230810024242_msg.nc
Missing: 20230810024242_msg.nc
Copied: 20230830094242_msg.nc
Missing: 20230830094242_msg.nc
Copied: 20230710232742_msg.nc
Missing: 20230710232742_msg.nc


## Run patching for msg files

## Delete the patches that are not common to af patches 
### af patches where created only when a fire pixel was detected inside the 32x32 pixel. for this reasonn the msg patches will be much more. 

In [7]:
import os
af_patches = "/mnt/data8tb/fire_detection/af_nasa_patched/2023"
msg_patches = "/mnt/data8tb/fire_detection/msg_patched/"

In [8]:
for file in os.listdir(msg_patches):
    if os.path.exists(os.path.join(af_patches, file)):
        continue
    else:
        os.remove(os.path.join(msg_patches, file))

In [10]:
for file in os.listdir(af_patches):
    if os.path.exists(os.path.join(msg_patches, file)):
        continue
    else:
        print(f"No file found for {file}")
        os.remove(os.path.join(af_patches, file))

No file found for 20230519015741_patch_308.tif
